**Linda Zier**

**ST 554**

**HW #9**

**Goal**

• Finding a data set you can fit supervised learning models with

• Using a numeric or binary response, fitting three different classes of models and choosing an overall best model.

• Writing a narrative (via a notebook) with explanations and discussions as you go through the above.

**Data**

The dataset I used gives insights into bike rentals in Seoul Korea based on environmental factors.


In [8]:
# read in the data and inspect
import pandas as pd
df = pd.read_csv('ST-554-repo/data/SeoulBikeData.csv', encoding='latin1')
print(df.head())
print(df.dtypes)
print(df.shape)


         Date  Rented Bike Count  Hour  Temperature(°C)  Humidity(%)  \
0  01/12/2017                254     0             -5.2           37   
1  01/12/2017                204     1             -5.5           38   
2  01/12/2017                173     2             -6.0           39   
3  01/12/2017                107     3             -6.2           40   
4  01/12/2017                 78     4             -6.0           36   

   Wind speed (m/s)  Visibility (10m)  Dew point temperature(°C)  \
0               2.2              2000                      -17.6   
1               0.8              2000                      -17.6   
2               1.0              2000                      -17.7   
3               0.9              2000                      -17.6   
4               2.3              2000                      -18.6   

   Solar Radiation (MJ/m2)  Rainfall(mm)  Snowfall (cm) Seasons     Holiday  \
0                      0.0           0.0            0.0  Winter  No Holiday   


**Setup**

First I have to setup everything. This includes installing pyspark, importing, and starting  my Spark Session.

In [11]:
# committing regularly
!cd ST-554-repo && git add -A && git commit -m "progress" && git push

On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean


In [ ]:
!find ~ -name "Zier_ST_554_HW9.ipynb" 2>/dev/null

!cp ~/Zier_ST_554_HW9.ipynb ~/ST-554-repo/Zier_ST_554_HW9.ipynb
!cd ST-554-repo && git add -A && git commit -m "progress" && git push

In [3]:
import pandas as pd
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/12 12:45:48 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/12 12:45:48 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [5]:
# clone my hub - just each first time I get started
#!git clone https://github.com/ljzier/ST-554-repo.git
!pip install pyspark


Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable


In [6]:

import pandas as pd
import pyspark.sql.functions as F
from pyspark.sql import SparkSession
from pyspark.sql.types import DoubleType

from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler

from pyspark.ml.regression import LinearRegression, RandomForestRegressor, GeneralizedLinearRegression
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder



Since I already loaded my data, now I clean and convert my dataframe to a Sparkdata frame.

In [9]:

#see if there are any non-functioning days
print(df['Functioning Day'].value_counts())
print(df[df['Functioning Day'] == 'No']['Rented Bike Count'].unique())

#filter out the non-functioning days and check the number of rows
df = df[df['Functioning Day'] == 'Yes']
print(df.shape)

#rename columns to make the names spark-friendly (no spaces)
df.columns = [
    'Date', 'Bike_Count', 'Hour', 'Temperature', 'Humidity',
    'Wind_Speed', 'Visibility', 'Dew_Point', 'Solar_Radiation',
    'Rainfall', 'Snowfall', 'Seasons', 'Holiday', 'Functioning_Day'
]

#convert to Spark data frame
sdf= spark.createDataFrame(df)

# month as new feature
sdf = sdf.withColumn('month', F.month(F.to_date(F.col('Date'), 'dd/MM/yyyy')))

# day of week as new feature
sdf = sdf.withColumn('day_of_week', F.dayofweek(F.to_date(F.col('Date'), 'dd/MM/yyyy')))

# make the type DoubleType which is like Float64
numeric_cols = ['Bike_Count', 'Hour', 'Temperature', 'Humidity', 'Wind_Speed', 
                'Visibility', 'Dew_Point', 'Solar_Radiation', 'Rainfall', 
                'Snowfall', 'Month', 'Day_of_Week']

for c in numeric_cols:
    sdf = sdf.withColumn(c, F.col(c).cast(DoubleType()))

sdf.printSchema()
sdf.show(5)

Functioning Day
Yes    8465
No      295
Name: count, dtype: int64
[0]
(8465, 14)
root
 |-- Date: string (nullable = true)
 |-- Bike_Count: double (nullable = true)
 |-- Hour: double (nullable = true)
 |-- Temperature: double (nullable = true)
 |-- Humidity: double (nullable = true)
 |-- Wind_Speed: double (nullable = true)
 |-- Visibility: double (nullable = true)
 |-- Dew_Point: double (nullable = true)
 |-- Solar_Radiation: double (nullable = true)
 |-- Rainfall: double (nullable = true)
 |-- Snowfall: double (nullable = true)
 |-- Seasons: string (nullable = true)
 |-- Holiday: string (nullable = true)
 |-- Functioning_Day: string (nullable = true)
 |-- Month: double (nullable = true)
 |-- Day_of_Week: double (nullable = true)



+----------+----------+----+-----------+--------+----------+----------+---------+---------------+--------+--------+-------+----------+---------------+-----+-----------+
|      Date|Bike_Count|Hour|Temperature|Humidity|Wind_Speed|Visibility|Dew_Point|Solar_Radiation|Rainfall|Snowfall|Seasons|   Holiday|Functioning_Day|Month|Day_of_Week|
+----------+----------+----+-----------+--------+----------+----------+---------+---------------+--------+--------+-------+----------+---------------+-----+-----------+
|01/12/2017|     254.0| 0.0|       -5.2|    37.0|       2.2|    2000.0|    -17.6|            0.0|     0.0|     0.0| Winter|No Holiday|            Yes| 12.0|        6.0|
|01/12/2017|     204.0| 1.0|       -5.5|    38.0|       0.8|    2000.0|    -17.6|            0.0|     0.0|     0.0| Winter|No Holiday|            Yes| 12.0|        6.0|
|01/12/2017|     173.0| 2.0|       -6.0|    39.0|       1.0|    2000.0|    -17.7|            0.0|     0.0|     0.0| Winter|No Holiday|            Yes| 12.0

#Splitting the Data, Metrics, and Models

• Using spark MLlib, split the data into a training and test set.

• Choose and describe a metric you’ll be using to judge your models.

• You’ll be fitting three different classes of models. Briefly describe each model

3 models I'll be fitting:
1.Linear Regression with Elastic Net
2.Random Forest Regressor

Random forest is an ML algorithm that combines the output of multiple decision trees to reach a single result. It handles both classification and regression problems.
3. Generalized Linear Regressor (Poisson)

3. Generalized Linear Regression (Poisson):
Regular linear regression assumes the response variable is continuous and normally distributed. But Rented Bike Count is a count that is always a whole number and can't be negative. Count data often follows a Poisson distribution instead of a normal one. Poisson regression handles this by modeling the log of the expected count as a linear combination of the predictors, which also guarantees predictions are always positive. 

#Model Fitting
Use Spark MLlib to fit your three different classes models to the training data. This
should be done using pipelines and cross validation to choose your best model for each model type. You
should compare your models using your metric chosen earlier.

• You should set up a pipeline in pyspark for each of your models

• You should do your transformations using the functions from MLlib to easily put them into the pipeline.

At least one of the pipelines should use four or more transformations prior to the model fit (estimator)

– VectorAssembler counts as a transformation

– Doing something like a log transform counts as well

– Adding polynomial terms or interaction terms counts

• You can use the same set of transformations for multiple models (if appropriate)

#Model Testing
Lastly, you should evaluate the best models from each class on the test set and state which overall model
was deemed the best.


In [10]:
from pyspark.ml import Pipeline

# Setup CrossValidator() object and then use the .fit() method on crossval
lr = LinearRegression()
paramGrid = ParamGridBuilder() \
    .addGrid(lr.regParam, [0, 0.5]) \
    .addGrid(lr.elasticNetParam, [0, 0.2]) \
    .build()
crossval = CrossValidator(estimator = lr,
                          estimatorParamMaps = paramGrid,
                          evaluator = RegressionEvaluator(metricName='rmse'),
                          numFolds=5)

# use pipeline to wrap transform, model, and predict into one easy call

pipeline = Pipeline(stages = [sqlTrans, assembler, lr])
crossval = CrossValidator(estimator = pipeline,
                          estimatorParamMaps = paramGrid,
                          evaluator = RegressionEvaluator(),
                          numFolds=5)
cvModel = crossval.fit(train)
cvModel.transform(test) #for predictions

NameError: name 'RegressionEvaluator' is not defined